# 4.4 LLM Read-only Bug Sweep

这个 notebook 用 4.3 的 LLM read-only agent 思路，批量测试多个 fixture workspaces。目标是观察 agent 是否能在不同小项目里定位潜在 bug，但仍然不修改任何文件。

## 目标

4.4 做一件事：对每个 bug workspace 运行一次只读 LLM debug agent，并用简单 heuristic 检查最终回答是否提到了预期 bug 线索。

它仍然不做：

- 文件修改
- patch
- shell command
- 自动修复

它只测试：LLM 能不能通过 read-only tools 找出 bug。

## 导入依赖

In [1]:
from __future__ import annotations

import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, cast

from dotenv import load_dotenv
from openai import APIConnectionError, APIStatusError, APITimeoutError, AuthenticationError, OpenAI
from openai.types.chat import (
    ChatCompletionMessageParam,
    ChatCompletionMessageToolCall,
    ChatCompletionToolParam,
)


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "prototype" else Path.cwd()
AGENT_SRC_ROOT = PROJECT_ROOT / "src"
WORKSPACES_ROOT = PROJECT_ROOT / "fixtures" / "workspaces"

if str(AGENT_SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(AGENT_SRC_ROOT))

import swe_agent_jom.config.settings as settings_module
import swe_agent_jom.runtime.workspace as workspace_module

from swe_agent_jom.core.tool_result import ToolResult, error_result
from swe_agent_jom.memory.trajectory import Trajectory
from swe_agent_jom.tools.file_tool import list_files, read_file
from swe_agent_jom.tools.search_tool import search_code

## 配置 LLM client 和连接测试

In [2]:
load_dotenv(PROJECT_ROOT / ".env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

if not DEEPSEEK_API_KEY:
    raise RuntimeError("Missing DEEPSEEK_API_KEY. Please add it to your .env file.")

MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash")
LLM_TIMEOUT_SECONDS = float(os.getenv("DEEPSEEK_TIMEOUT_SECONDS", "60"))
LLM_CONNECTIVITY_TIMEOUT_SECONDS = float(
    os.getenv("DEEPSEEK_CONNECTIVITY_TIMEOUT_SECONDS", "20")
)

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com",
    timeout=LLM_TIMEOUT_SECONDS,
)

print("MODEL:", MODEL)
print("TIMEOUT_SECONDS:", LLM_TIMEOUT_SECONDS)

MODEL: deepseek-v4-flash
TIMEOUT_SECONDS: 60.0


In [3]:
def check_llm_connectivity() -> None:
    """Fail fast on API/network/auth problems before running the sweep."""
    smoke_messages: list[ChatCompletionMessageParam] = [
        cast(ChatCompletionMessageParam, {"role": "system", "content": "Reply with OK."}),
        cast(ChatCompletionMessageParam, {"role": "user", "content": "ping"}),
    ]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=smoke_messages,
            max_tokens=8,
            timeout=LLM_CONNECTIVITY_TIMEOUT_SECONDS,
        )
    except APITimeoutError as error:
        raise RuntimeError(
            "LLM connectivity test timed out. Check network/proxy status, "
            "or increase DEEPSEEK_CONNECTIVITY_TIMEOUT_SECONDS."
        ) from error
    except APIConnectionError as error:
        raise RuntimeError(
            "LLM connectivity test failed to connect. Check DNS, proxy, or firewall."
        ) from error
    except AuthenticationError as error:
        raise RuntimeError(
            "LLM authentication failed. Check DEEPSEEK_API_KEY in .env."
        ) from error
    except APIStatusError as error:
        raise RuntimeError(
            f"LLM API returned HTTP {error.status_code}. Check model and service status."
        ) from error

    content = response.choices[0].message.content or ""
    print("LLM connectivity ok:", content[:80])


check_llm_connectivity()

LLM connectivity ok: 


## 定义 bug cases

每个 case 是一个独立 workspace。`expected_terms` 是一个轻量 heuristic，用来判断最终答案是否提到了关键 bug 线索。

In [4]:
@dataclass(frozen=True)
class BugCase:
    name: str
    workspace_path: Path
    task: str
    expected_terms: tuple[str, ...]


BUG_CASES: list[BugCase] = [
    BugCase(
        name="calculator_bug",
        workspace_path=WORKSPACES_ROOT / "calculator_bug",
        task="The calculator tests are failing. Inspect the workspace and identify the bug.",
        expected_terms=("add", "return a - b", "return a + b"),
    ),
    BugCase(
        name="string_utils_bug",
        workspace_path=WORKSPACES_ROOT / "string_utils_bug",
        task="The string utility tests are failing. Inspect the workspace and identify the bug.",
        expected_terms=("normalize_slug", "_", "-"),
    ),
    BugCase(
        name="list_filter_bug",
        workspace_path=WORKSPACES_ROOT / "list_filter_bug",
        task="The list filtering tests are failing. Inspect the workspace and identify the bug.",
        expected_terms=("only_even", "% 2 == 1", "% 2 == 0"),
    ),
    BugCase(
        name="config_parser_bug",
        workspace_path=WORKSPACES_ROOT / "config_parser_bug",
        task="The config parser tests are failing. Inspect the workspace and identify the bug.",
        expected_terms=("parse_bool", "false", "return False"),
    ),
]

## Read-only tool schemas

In [5]:
AVAILABLE_TOOL_NAMES = ["list_files", "read_file", "search_code"]


TOOLS: list[ChatCompletionToolParam] = [
    cast(ChatCompletionToolParam, {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files and directories inside the current target workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Workspace-relative path."},
                    "pattern": {"type": "string", "description": "Glob pattern, such as *.py."},
                    "max_results": {"type": "integer"},
                },
            },
        },
    }),
    cast(ChatCompletionToolParam, {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a text file inside the current target workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                    "max_chars": {"type": "integer"},
                },
                "required": ["path"],
            },
        },
    }),
    cast(ChatCompletionToolParam, {
        "type": "function",
        "function": {
            "name": "search_code",
            "description": "Search text inside files in the current target workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "path": {"type": "string"},
                    "pattern": {"type": "string"},
                    "case_sensitive": {"type": "boolean"},
                    "max_results": {"type": "integer"},
                },
                "required": ["query"],
            },
        },
    }),
]

## Workspace switching and tool dispatch

In [6]:
def set_target_workspace(workspace_path: Path) -> None:
    """Point read-only tools at one fixture workspace."""
    resolved = workspace_path.resolve()
    os.environ["SWE_AGENT_JOM_WORKSPACE_ROOT"] = str(resolved)
    settings_module.WORKSPACE_ROOT = resolved
    workspace_module.WORKSPACE_ROOT = resolved


def parse_tool_arguments(raw_arguments: str) -> dict[str, Any]:
    try:
        parsed = json.loads(raw_arguments or "{}")
    except json.JSONDecodeError as error:
        raise ValueError(f"Invalid JSON tool arguments: {error}") from error

    if not isinstance(parsed, dict):
        raise ValueError("Tool arguments must decode to a JSON object.")

    return cast(dict[str, Any], parsed)


def dispatch_tool(tool_name: str, arguments: dict[str, Any]) -> ToolResult:
    if tool_name == "list_files":
        return list_files(
            path=cast(str, arguments.get("path", ".")),
            pattern=cast(str, arguments.get("pattern", "*")),
            max_results=int(arguments.get("max_results", 50)),
        )

    if tool_name == "read_file":
        return read_file(
            path=cast(str, arguments["path"]),
            max_chars=int(arguments.get("max_chars", 3000)),
        )

    if tool_name == "search_code":
        return search_code(
            query=cast(str, arguments["query"]),
            path=cast(str, arguments.get("path", ".")),
            pattern=cast(str, arguments.get("pattern", "*")),
            case_sensitive=bool(arguments.get("case_sensitive", False)),
            max_results=int(arguments.get("max_results", 20)),
        )

    return error_result(
        "UnknownToolError",
        f"Unknown tool: {tool_name}",
        details={"available_tools": AVAILABLE_TOOL_NAMES},
    )


def tool_result_to_message(tool_call_id: str, result: ToolResult) -> ChatCompletionMessageParam:
    return cast(
        ChatCompletionMessageParam,
        {
            "role": "tool",
            "tool_call_id": tool_call_id,
            "content": json.dumps(result, ensure_ascii=False),
        },
    )

## Agent loop

In [7]:
def system_prompt_for_case(case: BugCase) -> str:
    return f"""
You are a read-only SWE debugging agent.

Current target workspace: fixtures/workspaces/{case.name}.

You can use these read-only tools:
- list_files
- read_file
- search_code

Rules:
1. Use tools to inspect the current target workspace before making claims.
2. Do not claim you modified files. You do not have write or patch tools.
3. Do not ask to run shell commands. Command execution is intentionally out of scope here.
4. If you identify a bug, explain the failing behavior, the relevant file, and the exact suggested code change.
5. Keep the final answer concise and practical.
""".strip()


def assistant_message_to_dict(message: Any) -> ChatCompletionMessageParam:
    return cast(ChatCompletionMessageParam, message.model_dump(exclude_none=True))


def run_case_agent(case: BugCase, *, max_turns: int = 8) -> dict[str, Any]:
    set_target_workspace(case.workspace_path)

    trajectory = Trajectory()
    trajectory.record_user_message(case.task)

    messages: list[ChatCompletionMessageParam] = [
        cast(ChatCompletionMessageParam, {"role": "system", "content": system_prompt_for_case(case)}),
        cast(ChatCompletionMessageParam, {"role": "user", "content": case.task}),
    ]

    for _ in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message_to_dict(assistant_message))

        content = assistant_message.content or ""
        if content:
            trajectory.record_assistant_message(content)

        tool_calls = assistant_message.tool_calls or []

        if not tool_calls:
            trajectory.record_final_answer(content)
            return {
                "case": case.name,
                "final_answer": content,
                "messages": messages,
                "trajectory": trajectory.to_dicts(),
            }

        for tool_call in tool_calls:
            function_tool_call = cast(ChatCompletionMessageToolCall, tool_call)
            tool_name = function_tool_call.function.name

            try:
                arguments = parse_tool_arguments(function_tool_call.function.arguments)
            except ValueError as error:
                result = error_result("InvalidToolArgumentsError", str(error))
                arguments = {"raw_arguments": function_tool_call.function.arguments}
            else:
                result = dispatch_tool(tool_name, arguments)

            trajectory.record_tool_call(tool_name, arguments, tool_call_id=function_tool_call.id)
            trajectory.record_tool_result(
                tool_name,
                cast(dict[str, Any], result),
                tool_call_id=function_tool_call.id,
            )
            messages.append(tool_result_to_message(function_tool_call.id, result))

    final_answer = "Stopped before final answer: max_turns reached."
    trajectory.record_final_answer(final_answer)
    return {
        "case": case.name,
        "final_answer": final_answer,
        "messages": messages,
        "trajectory": trajectory.to_dicts(),
    }

## Simple scoring

In [8]:
def score_case_result(case: BugCase, final_answer: str) -> dict[str, Any]:
    normalized_answer = final_answer.lower()
    matched_terms = [
        term for term in case.expected_terms if term.lower() in normalized_answer
    ]
    return {
        "case": case.name,
        "passed": len(matched_terms) == len(case.expected_terms),
        "matched_terms": matched_terms,
        "expected_terms": list(case.expected_terms),
        "final_answer": final_answer,
    }

## Run bug sweep

运行这个 cell 会对所有 fixture workspaces 调用模型 API。

In [9]:
sweep_results: list[dict[str, Any]] = []

for bug_case in BUG_CASES:
    print(f"Running case: {bug_case.name}")
    result = run_case_agent(bug_case)
    score = score_case_result(bug_case, cast(str, result["final_answer"]))
    sweep_results.append({"result": result, "score": score})
    print("passed:", score["passed"])
    print(cast(str, result["final_answer"])[:500])
    print("-" * 80)

Running case: calculator_bug
passed: True
## Bug Identified

**File:** `src/demo_math/calculator.py`

**Line 2:** The `add()` function has a sign error — it returns `a - b` instead of `a + b`.

```python
def add(a: int, b: int) -> int:
    return a - b   # ❌ BUG: subtraction instead of addition
```

This causes the test `test_adds_two_numbers` to fail because `add(2, 3)` returns `-1` instead of the expected `5`.

**Fix:** Change line 2 from `return a - b` to `return a + b`.

```python
def add(a: int, b: int) -> int:
    return a + b   #
--------------------------------------------------------------------------------
Running case: string_utils_bug
passed: True
## Bug Identified

In `src/text_tools/normalizer.py`, the `normalize_slug` function replaces spaces with underscores (`_`), but the test expects hyphens (`-`).

**File:** `src/text_tools/normalizer.py` (line 7)

**Current (buggy) code:**
```python
def normalize_slug(text: str) -> str:
    normalized = normalize_whitespace(text).lo

## Summary table

In [10]:
summary = [item["score"] for item in sweep_results]
print(json.dumps(summary, ensure_ascii=False, indent=2))

[
  {
    "case": "calculator_bug",
    "passed": true,
    "matched_terms": [
      "add",
      "return a - b",
      "return a + b"
    ],
    "expected_terms": [
      "add",
      "return a - b",
      "return a + b"
    ],
    "final_answer": "## Bug Identified\n\n**File:** `src/demo_math/calculator.py`\n\n**Line 2:** The `add()` function has a sign error — it returns `a - b` instead of `a + b`.\n\n```python\ndef add(a: int, b: int) -> int:\n    return a - b   # ❌ BUG: subtraction instead of addition\n```\n\nThis causes the test `test_adds_two_numbers` to fail because `add(2, 3)` returns `-1` instead of the expected `5`.\n\n**Fix:** Change line 2 from `return a - b` to `return a + b`.\n\n```python\ndef add(a: int, b: int) -> int:\n    return a + b   # ✅ correct\n```\n\nAll other functions (`subtract`, `multiply`, `divide`) look correct."
  },
  {
    "case": "string_utils_bug",
    "passed": true,
    "matched_terms": [
      "normalize_slug",
      "_",
      "-"
    ],
    "exp

## Inspect one trajectory

Change the index to inspect a specific case trajectory.

In [11]:
case_index = 0
print(json.dumps(sweep_results[case_index]["result"]["trajectory"], ensure_ascii=False, indent=2))

[
  {
    "kind": "user_message",
    "content": {
      "content": "The calculator tests are failing. Inspect the workspace and identify the bug."
    },
    "created_at": "2026-07-13T05:49:10.974379+00:00"
  },
  {
    "kind": "tool_call",
    "content": {
      "tool_call_id": "call_00_5iFP8rkvsmOjvaBFh5CW2585",
      "tool_name": "list_files",
      "arguments": {
        "path": "."
      }
    },
    "created_at": "2026-07-13T05:49:12.307721+00:00"
  },
  {
    "kind": "tool_result",
    "content": {
      "tool_call_id": "call_00_5iFP8rkvsmOjvaBFh5CW2585",
      "tool_name": "list_files",
      "result": {
        "ok": true,
        "data": {
          "path": ".",
          "pattern": "*",
          "count": 7,
          "max_results": 50,
          "truncated": false,
          "entries": [
            {
              "path": "README.md",
              "type": "file",
              "size_bytes": 405
            },
            {
              "path": "src",
              "type